<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_WorldFactCollector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

In [1]:
!pip -q install pymupdf
!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py                  # all
!wget -O mos1.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos1.py                  # all
import mos, mos1
from pathlib import Path
import time

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 25.8 MB/s eta 0:00:00


In [ ]:
import importlib
#importlib.reload(mos1)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:


WORLDFACTS = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/WorldFacts"

!ls -l "{WORLDFACTS}"

total 12
drwx------ 2 root root 4096 Sep 28 09:09 MD
drwx------ 2 root root 4096 Sep 29 09:42 NEWS
drwx------ 2 root root 4096 Sep 28 06:55 PDF


## Static World Data
URLs stored as PDF files that are converted to markdown files

In [4]:
client = mos.authenticateOpenAI()

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com


In [ ]:
# =============================================================================
# MOS STATIC SOURCE PACK v1
# =============================================================================

# 01. IEA — Oil Market Report, September 2026
# Global oil supply, demand, inventories, prices, refining and Gulf disruption.
# https://www.iea.org/reports/oil-market-report-september-2026

# 02. US EIA — Short-Term Energy Outlook: Global Oil Markets
# Oil prices, Middle East production/exports, Iran and Hormuz disruption.
# https://www.eia.gov/outlooks/steo/report/global_oil.php

# 03. PPAC — India's Oil & Gas Ready Reckoner / Current Reports
# India's crude production, imports, consumption, refining and import dependence.
# https://ppac.gov.in/
#Snapshot of India's Oil and Gas Data → Monthly Ready Reckoner — August 2026 → View Report (PDF)

# 04. Government of India / PIB — West Asia Energy Supply Disruption
# India's Gulf/Hormuz exposure and government energy-security response.
# https://www.pib.gov.in/PressReleasePage.aspx?PRID=2239021&lang=1&reg=6


# 06. World Bank — Iran Macro Poverty Outlook, April 2026
# Concise current picture of Iran's economy, sanctions, inflation, fiscal stress and oil vulnerability.
# https://documents.worldbank.org/en/publication/documents-reports/documentdetail/099211508192619381
# https://documents1.worldbank.org/curated/en/099211508192619381/pdf/Bulk-8dc686b3-acf3-4bd1-92b1-373dd5db8389.pdf

# 07. SIPRI — Yearbook 2026 Summary --- IGNORED
# Regional strategic context, military capabilities and nuclear forces.
# https://www.sipri.org/yearbook/2026/summary/sipri-yearbook-2026-summary-english

# 08. CSIS Missile Threat — Iran
# Iranian ballistic/cruise missiles, anti-ship systems and UAV capabilities.
# https://missilethreat.csis.org/country/iran/

# =============================================================================

In [ ]:
# 01. US Congressional Research Service (CRS) — U.S. Conflict with Iran
# Comprehensive overview of the 2026 US-Iran conflict, military operations, Hormuz, sanctions, diplomacy, regional actors and strategic issues.
# https://www.congress.gov/crs-product/R48887

# 02. UK House of Commons Library — Iran in 2026: Six Months After the US/Israeli Strikes
# Broad current assessment of the conflict, Iran's political/economic condition, Hormuz, sanctions, regional armed groups and US-Iran diplomacy.
# https://commonslibrary.parliament.uk/research-briefings/cbp-11075/

In [5]:
PDF_DIR = f"{WORLDFACTS}/PDF"
MD_DIR  = f"{WORLDFACTS}/MD"

!ls -l "{PDF_DIR}"

total 19124
-rw------- 1 root root   153646 Sep 28 21:39  Iran_Data.pdf
drwx------ 2 root root     4096 Oct  1 09:26  LessRelevent
-rw------- 1 root root  7057875 Sep 28 21:41 'Missiles of Iran _ Missile Threat.pdf'
-rw------- 1 root root   214516 Sep 28 03:04 'Oil Market Report - September 2026 – Analysis - IEA.pdf'
-rw------- 1 root root   259068 Sep 28 21:32 'Press Release Page _ Press Information Bureau.pdf'
-rw------- 1 root root   381243 Sep 28 21:28 'Short-Term Energy Outlook - U.S. Energy Information Administration (EIA).pdf'
-rw------- 1 root root 10479230 Sep 29 12:53  Snapshot_of_Indias_Oil_Gas_data.pdf
-rw------- 1 root root   175382 Sep 29 12:51 'UK-Commons-Iran in 2026_ Six months after the US_Israeli strikes.pdf'
-rw------- 1 root root   855750 Sep 29 12:51  US-Congress-R48887.3.pdf


In [6]:
Role_PDFReader = """
    You are a document-cleaning engine.

    Your task is to convert raw text extracted from a PDF into
    clean, well-structured Markdown.

    RULES

    1. Preserve ALL substantive factual content.
    2. Preserve all numbers, dates, quantities, percentages,
       names, units and factual claims.
    3. Preserve meaningful headings and section structure.
    4. Preserve useful tables. Reconstruct them as Markdown
       tables where reasonably possible.
    5. Remove advertisements, navigation menus, subscription
       prompts, cookie notices, social-media links, repeated
       headers and footers, page furniture and similar junk.
    6. Repair obvious line-break, hyphenation and formatting
       damage introduced by PDF extraction.
    7. Do NOT summarize.
    8. Do NOT interpret.
    9. Do NOT add information.
    10. Do NOT use outside knowledge.
    11. Do NOT correct or reconcile factual claims.
    12. If something is genuinely unclear or corrupted,
        preserve it rather than inventing content.
    13. If date of publication of document is known, state this at the top as date of publication.
        If date of publicatino of document is not clear, state date of publication not known. Do not guess

    Return ONLY the cleaned Markdown document.
    """

In [7]:
from pathlib import Path

pdf_files = sorted(Path(PDF_DIR).glob("*.pdf"))

# ------------------------------------------------------------------
# Delete old Markdown files
# ------------------------------------------------------------------

old_md_files = list(Path(MD_DIR).glob("*.md"))

for md_file in old_md_files:
    md_file.unlink()

print(f"Deleted {len(old_md_files)} old MD files")
print(f"Found {len(pdf_files)} PDF files\n")

pdf_files = sorted(Path(PDF_DIR).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files\n")

md_files = []
for source_PDF in pdf_files:
    print("=" * 70)
    print(f"Processing: {source_PDF.name}")
    try:
        md_path = mos1.GetWorldFile(client,Role_PDFReader,source_PDF,MD_DIR, _model="gpt-5-mini" )
        md_files.append(md_path)
    except Exception as e:
        print(f"❌ FAILED: {source_PDF.name}")
        print(f"Reason: {e}")

print("\n" + "=" * 70)
print(f"Completed: {len(md_files)} of {len(pdf_files)} files")

Deleted 9 old MD files
Found 8 PDF files

Found 8 PDF files

Processing: Iran_Data.pdf
Reading  : Iran_Data.pdf
Extracted: 5,504 characters
Model: gpt-5-mini-2025-08-07 | Effort: medium | Input: 2,973 | Output: 4,685 | Reasoning: 896 | Total: 7,658
Created  : IranData_261001.md
Markdown : 7,820 characters
Processing: Missiles of Iran _ Missile Threat.pdf
Reading  : Missiles of Iran _ Missile Threat.pdf
Extracted: 5,594 characters
Model: gpt-5-mini-2025-08-07 | Effort: medium | Input: 1,761 | Output: 2,658 | Reasoning: 1,216 | Total: 4,419
Created  : MissilesofIranMissil_261001.md
Markdown : 7,014 characters
Processing: Oil Market Report - September 2026 – Analysis - IEA.pdf
Reading  : Oil Market Report - September 2026 – Analysis - IEA.pdf
Extracted: 7,853 characters
Model: gpt-5-mini-2025-08-07 | Effort: medium | Input: 2,655 | Output: 4,031 | Reasoning: 1,664 | Total: 6,686
Created  : OilMarketReportSepte_261001.md
Markdown : 8,218 characters
Processing: Press Release Page _ Press In

In [ ]:
#print(md_files)

In [8]:
for f in md_files:
    print(f.name)

IranData_261001.md
MissilesofIranMissil_261001.md
OilMarketReportSepte_261001.md
PressReleasePagePres_261001.md
ShortTermEnergyOutlo_261001.md
SnapshotofIndiasOilG_261001.md
UKCommonsIranin2026S_261001.md
USCongressR488873_261001.md


In [10]:
#mos1.ShowMD(md_files[5])

## Live News Feed Collection and Storage
Currently only from MoneyControl

In [11]:
_mc_links = mos1.get_mc_links("https://www.moneycontrol.com/world/", "/world/")

/world/ links located : 57


In [12]:
stories = mos1.get_stories(_mc_links)

Processed 10/57 | Valid stories: 9
Processed 20/57 | Valid stories: 15
Processed 30/57 | Valid stories: 24
Processed 40/57 | Valid stories: 31
Processed 50/57 | Valid stories: 36
Processed 57/57 | Valid stories: 40
----------------------------------------
Links examined : 57
Valid stories  : 40


In [13]:
# ------------------------------------------------------------------------
# Score stories for Middle East Oil Security relevance
# ------------------------------------------------------------------------

def score_MOS(article):

    text = (
        article["title"]
        + " "
        + article["content"]
    ).lower()

    score = 0

    high = [
        "strait of hormuz",
        "hormuz",
        "iran",
        "iranian",
        "crude oil",
        "oil price",
        "brent",
        "middle east",
        "persian gulf"
    ]

    geopolitical = [
        "united states",
        "u.s.",
        "israel",
        "missile",
        "drone",
        "air strike",
        "airstrike",
        "attack",
        "war",
        "military",
        "sanctions",
        "retaliation",
        "ceasefire"
    ]

    oil_security = [
        "tanker",
        "shipping",
        "oil supply",
        "oil exports",
        "oil production",
        "refinery",
        "opec",
        "opec+",
        "saudi arabia",
        "uae",
        "qatar",
        "gulf"
    ]

    india = [
        "india",
        "indian oil",
        "ioc",
        "bharat petroleum",
        "hindustan petroleum",
        "reliance industries",
        "oil import",
        "petroleum ministry"
    ]

    for k in high:
        if k in text:
            score += 4

    for k in geopolitical:
        if k in text:
            score += 2

    for k in oil_security:
        if k in text:
            score += 2

    for k in india:
        if k in text:
            score += 2

    return score

In [14]:
# ------------------------------------------------------------------------
# Score and sort stories by MOS relevance
# ------------------------------------------------------------------------

for s in stories:
    s["score"] = score_MOS(s)

stories = sorted(
    stories,
    key=lambda x: x["score"],
    reverse=True
)

for i, s in enumerate(stories, 1):
    print(f"{i:3}. [{s['score']:2}] {s['title']}")

  1. [22] Iran-link in flydubai attack? Netanyahu says Israel will 'get to the root' of Omani co-pilot
  2. [18] Netanyahu accuses UK of 'caving in' after Israel shared intel on alleged Iran-backed attack
  3. [16] Japan's biggest manufacturers most optimistic since 2018
  4. [12] Taliban turns up heat on Pakistan: Noori warns Asim Munir, claims 'we can raise Afghan flag in Peshawar in 2 days'
  5. [12] Israel to ramp up pilot checks after mid-air altercation in flydubai flight
  6. [10] Cockpit clash, then a 17,000-ft plunge: Terrifying blow-by-blow account of flydubai flight horror visualised
  7. [10] flydubai suspends flights to and from Israel after mid-air pilot altercation on Dubai-Tel Aviv plane
  8. [10] flydubai mid-air horror: Passenger credits Indian pilot Smit Machchhar with saving lives after cockpit attack
  9. [10] What is flydubai and why it matters to Israelis: All about the airline in spotlight after hijack scare
 10. [ 8] How did a knife reach flydubai’s cockpit? Th

In [15]:
# ------------------------------------------------------------------------
# Inspect top MOS-ranked stories
# ------------------------------------------------------------------------

for i, s in enumerate(stories[:15], 1):

    print("=" * 80)
    print(f"{i}. SCORE: {s['score']}")
    print(s["title"])
    print()
    print(s["content"])
    print()

1. SCORE: 22
Iran-link in flydubai attack? Netanyahu says Israel will 'get to the root' of Omani co-pilot

Israeli Prime Minister Benjamin Netanyahu said on Wednesday that Israel will get to the "root" of the co-pilot who engaged in a foiled attempt to crash an Israel-bound flydubai flight and that Israel will "possibly" be participating in the probe into the incident. Speaking in an interview with Fox News following the attempted attack on flydubai Flight FZ1073, Israeli Prime Minister Benjamin Netanyahu addressed whether Iran was involved in the incident, stating, "It's too early to say." Netanyahu said the attacking pilot was being investigated in Saudi Arabia and could be taken to the UAE, adding that Israel would follow the investigation and could potentially participate in it. Why Iran has come at the centre of incident? Amid the ongoing Gulf war that erupted after US and Israel forces launched a joint attack on Iranian military infrastructure and leadership, Netanyahu believes t

In [16]:
live_file = mos1.CreateLiveFeed(stories, 4)

Stories selected : 30
File created     : MOS-MC-LiveFeed_261001_1509_IST.txt


In [17]:
!ls -l "{WORLDFACTS}/NEWS"

total 50
-rw------- 1 root root 50844 Sep 29 13:54 MOS-MC-LiveFeed_260929_1924_IST.txt


In [18]:
!rm "{WORLDFACTS}"/NEWS/*.txt

In [19]:
!ls -l "{WORLDFACTS}"
NEWS_DIR = f"{WORLDFACTS}/NEWS"

!ls -l "{WORLDFACTS}/NEWS"
!ls -l "{WORLDFACTS}/MD"

total 12
drwx------ 2 root root 4096 Oct  1 09:37 MD
drwx------ 2 root root 4096 Oct  1 09:43 NEWS
drwx------ 3 root root 4096 Sep 28 06:55 PDF
total 0
total 159
-rw------- 1 root root  7837 Oct  1 09:31 IranData_261001.md
-rw------- 1 root root  7031 Oct  1 09:31 MissilesofIranMissil_261001.md
-rw------- 1 root root  8238 Oct  1 09:32 OilMarketReportSepte_261001.md
-rw------- 1 root root 12434 Oct  1 09:32 PressReleasePagePres_261001.md
-rw------- 1 root root  4324 Oct  1 09:33 ShortTermEnergyOutlo_261001.md
-rw------- 1 root root 30798 Oct  1 09:34 SnapshotofIndiasOilG_261001.md
-rw------- 1 root root 17961 Oct  1 09:35 UKCommonsIranin2026S_261001.md
-rw------- 1 root root 71080 Oct  1 09:37 USCongressR488873_261001.md


In [20]:
!mv "{live_file}" "{NEWS_DIR}/"

In [21]:
!ls -l "{MD_DIR}"
!ls -l "{NEWS_DIR}"

total 159
-rw------- 1 root root  7837 Oct  1 09:31 IranData_261001.md
-rw------- 1 root root  7031 Oct  1 09:31 MissilesofIranMissil_261001.md
-rw------- 1 root root  8238 Oct  1 09:32 OilMarketReportSepte_261001.md
-rw------- 1 root root 12434 Oct  1 09:32 PressReleasePagePres_261001.md
-rw------- 1 root root  4324 Oct  1 09:33 ShortTermEnergyOutlo_261001.md
-rw------- 1 root root 30798 Oct  1 09:34 SnapshotofIndiasOilG_261001.md
-rw------- 1 root root 17961 Oct  1 09:35 UKCommonsIranin2026S_261001.md
-rw------- 1 root root 71080 Oct  1 09:37 USCongressR488873_261001.md
total 56
-rw------- 1 root root 57175 Oct  1 09:39 MOS-MC-LiveFeed_261001_1509_IST.txt


In [ ]:
#Check balance
# https://platform.openai.com/usage?utm_source=chatgpt.com